# M15 G4 — evaluación en el test de G3T (§4–§5 de `docs/experiments/M15_G4_prior.md`)

Notebook escrito **antes** de tener las predicciones de G4-R y G4-F (§6, paso 5). Se ejecuta una sola vez.

**Entradas (solo lectura; rutas resueltas desde las configs de predicción, regla de `scripts/predict_cnn_hdf5.py`:
`<data_root>/results/<dataset_id>/<output.file_name>`):**

- 5a: `configs/prediction/predict_m14_5a_4aG3T_fullcov_s123_on_G3T_test.json`
- G4-R: `configs/prediction/predict_m15_G4R_s123_on_G3T_test.json`
- G4-F: `configs/prediction/predict_m15_G4F_s123_on_G3T_test.json`
- De cada npz: `pred_test` (μ̂, la media de la cabeza), `y_test`, `idx_test`, `y_mean`, `y_std`. `pred_test` e `y_test`
  están estandarizados con el escalador train-only; físicas: x·y_std + y_mean.
- Cortes: cuantiles de la verdad en `processed/M13_G3_band1024_w16_500k/m13_13_g3t/fp16_memmap/y_train.npy`
  (400k, sin estandarizar; `np.quantile`, interpolación lineal por defecto), comparados con los del paso 0 (§5.1) con
  tolerancia 1e-3.

**Definiciones (§4):** b_X,g = media en g de (μ̂_X − θ) para la etiqueta primaria (M☉), con error típico s/√n.
D_g(X, Y) = |b_Y,g| − |b_X,g| (D > 0: X tiene menos sesgo). ΔRMSE_rel = RMSE_X / RMSE_Y − 1 en I.

**Bootstrap (§4):** emparejado por fila, 2000 réplicas, semilla 151001, percentiles 2,5 / 97,5.
Detalle de implementación (no lo fija el pre-registro): para cada grupo se crea un `np.random.default_rng(151001)`
nuevo y se extraen las 2000 réplicas en secuencia (`rng.integers(0, n_g, n_g)` por réplica); los mismos índices
remuestreados se aplican a los tres modelos, de modo que todas las comparaciones de un grupo comparten réplicas.

**Secundarias (§5.5):** la curva b(θ) usa 20 bins con bordes en los cuantiles de la verdad en G3T train (los mismos
datos que fijan los cortes de §4); las filas de test fuera del rango de train caen en los bins extremos.

In [ ]:
import hashlib
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root

PRED_CFGS = {
    '5a': 'configs/prediction/predict_m14_5a_4aG3T_fullcov_s123_on_G3T_test.json',
    'G4-R': 'configs/prediction/predict_m15_G4R_s123_on_G3T_test.json',
    'G4-F': 'configs/prediction/predict_m15_G4F_s123_on_G3T_test.json',
}
MODELS = tuple(PRED_CFGS)
LABELS = ('chirp_mass', 'total_mass', 'chi_eff')   # orden del contrato
LABEL_TXT = ('𝓜', 'M_tot', 'χ_eff')
N_TEST = 30000

CFGS = {m: json.loads((ROOT / p).read_text()) for m, p in PRED_CFGS.items()}
# Misma convención que los notebooks de M13/M15: CBC_PE_DATA_ROOT si está definida, si no el data_root de la config.
DATA_ROOT = resolve_data_root(config_data_root=CFGS['5a'].get('data_root'))
dataset_ids = {c['dataset']['dataset_id'] for c in CFGS.values()}
assert len(dataset_ids) == 1, dataset_ids
DATASET_ID = dataset_ids.pop()
for m, c in CFGS.items():
    assert c['prediction']['splits'] == ['test'], (m, c['prediction']['splits'])
    for k in ('dataset_file', 'split_file', 'label_stats_file'):
        assert c['dataset'][k] == CFGS['5a']['dataset'][k], (m, k)
    assert c['prediction']['memmap_dir'] == CFGS['5a']['prediction']['memmap_dir'], m

# scripts/predict_cnn_hdf5.py: output_path = resolve_path(<data_root>/results/<dataset_id>, output.file_name)
NPZ = {m: (DATA_ROOT / 'results' / DATASET_ID / c['output']['file_name']).resolve() for m, c in CFGS.items()}
Y_TRAIN = DATA_ROOT / 'processed' / DATASET_ID / 'fp16_memmap' / 'y_train.npy'
OUT_DIR = DATA_ROOT / 'results' / 'M13_3' / 'M15'

print('DATA_ROOT :', DATA_ROOT)
print('DATASET_ID:', DATASET_ID)
for m in MODELS:
    print(f'{m:5s} {NPZ[m]}  exists={NPZ[m].exists()}')
print('y_train   :', Y_TRAIN, ' exists=', Y_TRAIN.exists())
assert all(p.exists() for p in NPZ.values()) and Y_TRAIN.exists()

## Carga, desestandarización y alineación

Se comprueba que `idx_test` es idéntico en los tres npz (y que `y_test`, `y_mean`, `y_std` y `label_names` coinciden).

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


RAW, PROV = {}, {}
for m in MODELS:
    with np.load(NPZ[m], allow_pickle=False) as z:
        RAW[m] = {k: np.asarray(z[k]) for k in ('pred_test', 'y_test', 'idx_test', 'y_mean', 'y_std', 'label_names')}
        PROV[m] = {k: str(z[k]) for k in ('checkpoint_file', 'dataset_path', 'split_path', 'label_stats_path')}
    PROV[m]['npz'] = str(NPZ[m])
    PROV[m]['npz_sha256'] = sha256_file(NPZ[m])
    PROV[m]['prediction_config'] = PRED_CFGS[m]

ref = RAW['5a']
for m in MODELS:
    r = RAW[m]
    assert tuple(str(s) for s in r['label_names']) == LABELS, (m, r['label_names'])
    assert r['pred_test'].shape == r['y_test'].shape == (N_TEST, 3), (m, r['pred_test'].shape, r['y_test'].shape)
    assert r['idx_test'].shape == (N_TEST,) and np.unique(r['idx_test']).size == N_TEST, m
    assert np.array_equal(r['idx_test'], ref['idx_test']), f'idx_test de {m} difiere del de 5a'
    assert np.array_equal(r['y_mean'], ref['y_mean']) and np.array_equal(r['y_std'], ref['y_std']), m
    dy = np.max(np.abs(r['y_test'].astype(np.float64) - ref['y_test'].astype(np.float64)))
    print(f'{m:5s} idx_test idéntico a 5a: True   max|Δy_test| (estandarizado) = {dy:.3g}')
    assert dy <= 1e-6, m

Y_MEAN = ref['y_mean'].astype(np.float64)
Y_STD = ref['y_std'].astype(np.float64)
IDX = ref['idx_test']
Y = ref['y_test'].astype(np.float64) * Y_STD + Y_MEAN                       # verdad física (N, 3)
MU = {m: RAW[m]['pred_test'].astype(np.float64) * Y_STD + Y_MEAN for m in MODELS}
E = {m: MU[m] - Y for m in MODELS}                                          # μ̂ − θ (N, 3)
print('y_mean:', Y_MEAN, ' y_std:', Y_STD)
for m in MODELS:
    print(m, PROV[m]['checkpoint_file'])

## Cortes (cuantiles de la verdad en G3T train) frente al paso 0

In [ ]:
y_train = np.load(Y_TRAIN, mmap_mode='r')   # sin estandarizar (scripts/m13/train_m13_3.py)
assert y_train.shape == (400000, 3), y_train.shape
MC_TR = np.asarray(y_train[:, 0], dtype=np.float64)
MT_TR = np.asarray(y_train[:, 1], dtype=np.float64)

CUTS = {
    'Q95_mc': float(np.quantile(MC_TR, 0.95)),
    'Q05_mc': float(np.quantile(MC_TR, 0.05)),
    'Q95_mt': float(np.quantile(MT_TR, 0.95)),
    'Q10_mc': float(np.quantile(MC_TR, 0.10)),
    'Q90_mc': float(np.quantile(MC_TR, 0.90)),
    'Q10_mt': float(np.quantile(MT_TR, 0.10)),
    'Q90_mt': float(np.quantile(MT_TR, 0.90)),
}
# §5.1, resultado del paso 0 (20-11-2026)
PREREG_CUTS = {'Q95_mc': 66.259, 'Q05_mc': 13.287, 'Q95_mt': 153.085, 'Q10_mc': 16.328,
               'Q90_mc': 61.049, 'Q10_mt': 47.967, 'Q90_mt': 142.077}
CUT_TOL = 1e-3
for k in PREREG_CUTS:
    d = CUTS[k] - PREREG_CUTS[k]
    print(f'{k}: {CUTS[k]:.6f}   pre-registro {PREREG_CUTS[k]:.3f}   Δ = {d:+.2e}')
    assert abs(d) <= CUT_TOL, f'{k} no coincide con el pre-registro'
print('Cortes coinciden con el pre-registro (tolerancia 1e-3).')

## (1) Grupos de §4

| grupo | definición | etiqueta primaria |
|---|---|---|
| B_𝓜+ | 𝓜 > Q95(𝓜) | 𝓜 |
| B_Mt+ | M_tot > Q95(M_tot) | M_tot |
| B_𝓜− | 𝓜 < Q05(𝓜) | 𝓜 |
| I | Q10 ≤ 𝓜 ≤ Q90 y Q10 ≤ M_tot ≤ Q90 | las tres |

In [ ]:
MC, MT = Y[:, 0], Y[:, 1]
GROUPS = {   # nombre -> (máscara sobre las filas de test, índice de la etiqueta primaria o None)
    'B_Mc+': (MC > CUTS['Q95_mc'], 0),
    'B_Mt+': (MT > CUTS['Q95_mt'], 1),
    'B_Mc-': (MC < CUTS['Q05_mc'], 0),
    'I': ((MC >= CUTS['Q10_mc']) & (MC <= CUTS['Q90_mc']) & (MT >= CUTS['Q10_mt']) & (MT <= CUTS['Q90_mt']), None),
}
BORDER = ('B_Mc+', 'B_Mt+', 'B_Mc-')
GROUP_TXT = {'B_Mc+': 'B_𝓜+', 'B_Mt+': 'B_Mt+', 'B_Mc-': 'B_𝓜−', 'I': 'I'}
N_G = {g: int(mask.sum()) for g, (mask, _) in GROUPS.items()}
for g, (mask, k) in GROUPS.items():
    prim = LABEL_TXT[k] if k is not None else 'las tres'
    print(f'{GROUP_TXT[g]:6s} n = {N_G[g]:6d}   primaria: {prim}')

## (2) Sesgo b (M☉) de la etiqueta primaria por modelo y grupo, con su error típico

In [ ]:
def bias_se(e):
    # media y error típico (ddof = 1) por columna
    return e.mean(axis=0), e.std(axis=0, ddof=1) / np.sqrt(e.shape[0])


B, SE = {}, {}   # B[m][g] -> (3,) sesgo de las tres etiquetas en g
for m in MODELS:
    B[m], SE[m] = {}, {}
    for g, (mask, _) in GROUPS.items():
        B[m][g], SE[m][g] = bias_se(E[m][mask])

print(f"{'modelo':6s}" + ''.join(f'{GROUP_TXT[g] + " (n " + str(N_G[g]) + ")":>24s}' for g in BORDER))
for m in MODELS:
    row = ''.join(f'{B[m][g][GROUPS[g][1]]:+12.3f} ± {SE[m][g][GROUPS[g][1]]:.3f}  ' for g in BORDER)
    print(f'{m:6s}{row}')

## Bootstrap emparejado por fila (común a (3), (4) y (6))

Para cada grupo: 2000 réplicas, `default_rng(151001)` reiniciado por grupo, mismos índices para los tres modelos.
Se guardan por réplica el sesgo y el MSE de las tres etiquetas de cada modelo.

In [ ]:
N_BOOT, BOOT_SEED = 2000, 151001
PCT = (2.5, 97.5)

BOOT = {}
for g, (mask, _) in GROUPS.items():
    n = N_G[g]
    errs = {m: E[m][mask] for m in MODELS}
    bias_b = {m: np.empty((N_BOOT, 3)) for m in MODELS}
    mse_b = {m: np.empty((N_BOOT, 3)) for m in MODELS}
    rng = np.random.default_rng(BOOT_SEED)
    for b in range(N_BOOT):
        ii = rng.integers(0, n, size=n)
        for m in MODELS:
            e = errs[m][ii]
            bias_b[m][b] = e.mean(axis=0)
            mse_b[m][b] = np.mean(e * e, axis=0)
    BOOT[g] = {'bias': bias_b, 'mse': mse_b}
    print(f'{GROUP_TXT[g]:6s} bootstrap hecho (n = {n})')

## (3) Diferencias emparejadas D_g y veredicto (§5.2)

- G4-R frente a 5a en B_𝓜+ y B_Mt+ (H-RANGO); G4-F frente a G4-R en B_𝓜− (H-FORMA).
- **RELEVANTE:** D̂ ≥ δ y extremo inferior del IC > 0. **NO RELEVANTE:** extremo superior del IC < δ.
  **INDETERMINADO:** en otro caso.

In [ ]:
# δ_g = ½ |b_ref,g^val|, fijados en el paso 0 (§5.1, tabla «Criterios derivados»)
DELTA = {'B_Mc+': 2.040, 'B_Mt+': 3.970, 'B_Mc-': 0.112}
# Certificabilidad con una semilla según el paso 0 (§5.1); B_𝓜− en el límite (δ / 2 s_mod = 1,04)
CERTIFIABLE = {'B_Mc+': True, 'B_Mt+': True, 'B_Mc-': True}
PRIMARY_TESTS = [('B_Mc+', 'G4-R', '5a'), ('B_Mt+', 'G4-R', '5a'), ('B_Mc-', 'G4-F', 'G4-R')]   # (g, X, Y)


def paired_D(g, X, Yc):
    k = GROUPS[g][1]
    d_hat = abs(B[Yc][g][k]) - abs(B[X][g][k])
    d_boot = np.abs(BOOT[g]['bias'][Yc][:, k]) - np.abs(BOOT[g]['bias'][X][:, k])
    lo, hi = np.percentile(d_boot, PCT)
    return float(d_hat), float(lo), float(hi)


def relevance_verdict(d_hat, lo, hi, delta):
    if d_hat >= delta and lo > 0:
        return 'RELEVANTE'
    if hi < delta:
        return 'NO RELEVANTE'
    return 'INDETERMINADO'


PRIMARY = {}
print(f"{'grupo':6s} {'X':5s} {'Y':5s} {'|b_X|':>8s} {'|b_Y|':>8s} {'D̂':>8s} {'IC95':>20s} {'δ':>7s}  veredicto")
for g, X, Yc in PRIMARY_TESTS:
    k = GROUPS[g][1]
    d_hat, lo, hi = paired_D(g, X, Yc)
    v = relevance_verdict(d_hat, lo, hi, DELTA[g])
    PRIMARY[g] = {'X': X, 'Y': Yc, 'label': LABELS[k], 'n': N_G[g],
                  'b_X': float(B[X][g][k]), 'se_X': float(SE[X][g][k]),
                  'b_Y': float(B[Yc][g][k]), 'se_Y': float(SE[Yc][g][k]),
                  'D_hat': d_hat, 'ci95': [lo, hi], 'delta': DELTA[g], 'verdict': v}
    print(f'{GROUP_TXT[g]:6s} {X:5s} {Yc:5s} {abs(B[X][g][k]):8.3f} {abs(B[Yc][g][k]):8.3f} {d_hat:+8.3f} '
          f'[{lo:+8.3f}, {hi:+8.3f}] {DELTA[g]:7.3f}  {v}')

## (4) Coste en el interior I: no inferioridad (§5.3)

ΔRMSE_rel = RMSE_X / RMSE_Y − 1 por etiqueta, IC por el mismo bootstrap. Márgenes 3 / 3 / 5 %.
**NO INFERIOR:** extremo superior ≤ margen. **INFERIOR:** extremo inferior > margen. **INDETERMINADO:** en otro caso.

In [ ]:
MARGIN = (0.03, 0.03, 0.05)   # §5.3: 𝓜, M_tot, χ_eff
NI_TESTS = [('G4-R', '5a'), ('G4-F', 'G4-R')]   # (X, Y)


def ni_verdict(lo, hi, margin):
    if hi <= margin:
        return 'NO INFERIOR'
    if lo > margin:
        return 'INFERIOR'
    return 'INDETERMINADO'


mask_I = GROUPS['I'][0]
RMSE_I = {m: np.sqrt(np.mean(E[m][mask_I] ** 2, axis=0)) for m in MODELS}
COST = {}
print(f"n(I) = {N_G['I']}")
for X, Yc in NI_TESTS:
    key = f'{X} vs {Yc}'
    COST[key] = {}
    ratio_b = np.sqrt(BOOT['I']['mse'][X] / BOOT['I']['mse'][Yc]) - 1.0   # (N_BOOT, 3)
    for k, lab in enumerate(LABELS):
        r_hat = float(RMSE_I[X][k] / RMSE_I[Yc][k] - 1.0)
        lo, hi = (float(v) for v in np.percentile(ratio_b[:, k], PCT))
        v = ni_verdict(lo, hi, MARGIN[k])
        COST[key][lab] = {'rmse_X': float(RMSE_I[X][k]), 'rmse_Y': float(RMSE_I[Yc][k]),
                          'dRMSE_rel': r_hat, 'ci95': [lo, hi], 'margin': MARGIN[k], 'verdict': v}
        print(f'{key:12s} {LABEL_TXT[k]:6s} RMSE {RMSE_I[X][k]:8.4f} / {RMSE_I[Yc][k]:8.4f}   '
              f'ΔRMSE_rel = {100 * r_hat:+6.2f} %  IC [{100 * lo:+6.2f}, {100 * hi:+6.2f}] %  '
              f'margen {100 * MARGIN[k]:.0f} %  {v}')

## (5) Decisiones (§5.4)

- **H-RANGO** (G4-R frente a 5a; B_𝓜+ y B_Mt+, unión-intersección sin corrección por multiplicidad): APOYADA si los dos
  dan RELEVANTE; REFUTADA si los dos dan NO RELEVANTE; en otro caso, mixto o indeterminado, se informa tal cual.
- **H-FORMA** (G4-F frente a G4-R; B_𝓜−): APOYADA si RELEVANTE; REFUTADA si NO RELEVANTE; si no, INDETERMINADA.
- **Coste:** si una hipótesis queda apoyada con coste INFERIOR en alguna etiqueta, el compromiso se documenta y la
  adopción se decide en el chat (no es automática).
- Los tres grupos primarios fueron certificables en el paso 0, así que ninguna fila pasa a descriptiva.

In [ ]:
WARNING_B_MC_MINUS = (
    'Advertencia pre-registrada (§5.1): B_𝓜− es certificable en el límite (δ / 2 s_mod = 1,04). Con los seis modelos '
    'de G3T, s_mod = 0,093 y B_𝓜− no sería certificable (2 × 0,093 = 0,186 > δ = 0,112): el efecto relevante '
    '(0,11 M☉) es del orden de la variación entre modelos. Cualquier conclusión sobre H-FORMA se informa con esta advertencia.'
)


def hypothesis_decision(verdicts, n_required):
    if all(v == 'RELEVANTE' for v in verdicts) and len(verdicts) == n_required:
        return 'APOYADA'
    if all(v == 'NO RELEVANTE' for v in verdicts):
        return 'REFUTADA'
    return 'MIXTO / INDETERMINADO (se informa tal cual)' if n_required > 1 else 'INDETERMINADA'


DECISIONS = {}
for name, groups, cost_key in (('H-RANGO', ('B_Mc+', 'B_Mt+'), 'G4-R vs 5a'), ('H-FORMA', ('B_Mc-',), 'G4-F vs G4-R')):
    verdicts = [PRIMARY[g]['verdict'] for g in groups]
    dec = hypothesis_decision(verdicts, len(groups))
    if not all(CERTIFIABLE[g] for g in groups):
        dec = f'DESCRIPTIVA (grupo no certificable en el paso 0): {dec}'
    cost = {lab: COST[cost_key][lab]['verdict'] for lab in LABELS}
    inferior = [lab for lab, v in cost.items() if v == 'INFERIOR']
    if dec == 'APOYADA' and inferior:
        cost_note = f'apoyada con coste INFERIOR en {inferior}: compromiso documentado; adopción en el chat'
    elif inferior:
        cost_note = f'coste INFERIOR en {inferior}'
    else:
        cost_note = 'sin coste INFERIOR'
    DECISIONS[name] = {'comparison': cost_key, 'groups': list(groups),
                       'group_verdicts': dict(zip(groups, verdicts)), 'decision': dec,
                       'cost_I': cost, 'cost_note': cost_note}
    if name == 'H-FORMA':
        DECISIONS[name]['warning'] = WARNING_B_MC_MINUS

print(f"{'hipótesis':9s} {'comparación':13s} {'grupos y veredictos':45s} decisión")
for name, d in DECISIONS.items():
    gv = ', '.join(f'{GROUP_TXT[g]}: {v}' for g, v in d['group_verdicts'].items())
    print(f"{name:9s} {d['comparison']:13s} {gv:45s} {d['decision']}")
    print(f"{'':9s} coste en I: " + ', '.join(f'{LABEL_TXT[k]} {d["cost_I"][lab]}' for k, lab in enumerate(LABELS))
          + f"  -> {d['cost_note']}")
print()
print(WARNING_B_MC_MINUS)

## (6) Secundarias descriptivas (§5.5, sin criterio)

### 6a. Curva b(θ) en 20 bins de cuantiles de la verdad (bordes en G3T train)

Punto en la mediana de la verdad del bin; barras = error típico. Líneas verticales: cortes de §4.

In [ ]:
N_BINS = 20
CURVES = {}
for k, lab in enumerate(LABELS):
    edges = np.quantile(np.asarray(y_train[:, k], dtype=np.float64), np.linspace(0.0, 1.0, N_BINS + 1))
    which = np.clip(np.searchsorted(edges[1:-1], Y[:, k], side='right'), 0, N_BINS - 1)
    x_med = np.array([np.median(Y[which == j, k]) for j in range(N_BINS)])
    n_bin = np.bincount(which, minlength=N_BINS)
    CURVES[lab] = {'edges': edges.tolist(), 'x_median': x_med.tolist(), 'n': n_bin.tolist()}
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for m in MODELS:
        b = np.array([E[m][which == j, k].mean() for j in range(N_BINS)])
        se = np.array([E[m][which == j, k].std(ddof=1) / np.sqrt(n_bin[j]) for j in range(N_BINS)])
        CURVES[lab][m] = {'b': b.tolist(), 'se': se.tolist()}
        ax.errorbar(x_med, b, yerr=se, marker='o', ms=3.5, capsize=2, lw=1.2, label=m)
    ax.axhline(0.0, color='0.4', lw=0.8)
    if k == 0:
        for c in ('Q05_mc', 'Q10_mc', 'Q90_mc', 'Q95_mc'):
            ax.axvline(CUTS[c], color='0.6', ls=':', lw=0.9)
    elif k == 1:
        for c in ('Q10_mt', 'Q90_mt', 'Q95_mt'):
            ax.axvline(CUTS[c], color='0.6', ls=':', lw=0.9)
    unit = '' if k == 2 else ' (M☉)'
    ax.set_xlabel(f'{LABEL_TXT[k]} verdadera{unit}')
    ax.set_ylabel(f'b = media(μ̂ − θ){unit}')
    ax.set_title(f'Sesgo condicionado b(θ), {LABEL_TXT[k]} — test de G3T ({N_BINS} bins de cuantiles de train)')
    ax.legend(loc='best', frameon=True)
    fig.tight_layout()
    plt.show()

### 6b. MSE en los grupos de borde (las tres etiquetas)

Se espera que G4 empeore el MSE en el test de G3T (§5.5): un MSE peor no es un fallo.

In [ ]:
MSE_BORDER = {}
print(f"{'grupo':6s} {'modelo':6s}" + ''.join(f'{"MSE " + t:>16s}' for t in LABEL_TXT) + ''.join(f'{"/5a " + t:>12s}' for t in LABEL_TXT))
for g in BORDER:
    mask = GROUPS[g][0]
    MSE_BORDER[g] = {m: np.mean(E[m][mask] ** 2, axis=0).tolist() for m in MODELS}
    for m in MODELS:
        mse = np.array(MSE_BORDER[g][m])
        rel = mse / np.array(MSE_BORDER[g]['5a'])
        print(f'{GROUP_TXT[g]:6s} {m:6s}' + ''.join(f'{v:16.4f}' for v in mse) + ''.join(f'{v:12.3f}' for v in rel))

### 6c. χ_eff en los grupos de borde (sesgo ± error típico y RMSE)

In [ ]:
CHI_BORDER = {}
for g in BORDER:
    mask = GROUPS[g][0]
    CHI_BORDER[g] = {}
    for m in MODELS:
        e = E[m][mask, 2]
        CHI_BORDER[g][m] = {'b': float(e.mean()), 'se': float(e.std(ddof=1) / np.sqrt(e.size)),
                            'rmse': float(np.sqrt(np.mean(e * e)))}
        d = CHI_BORDER[g][m]
        print(f"{GROUP_TXT[g]:6s} {m:6s} b = {d['b']:+.4f} ± {d['se']:.4f}   RMSE = {d['rmse']:.4f}")

### 6d. G4-F frente a G4-R en B_𝓜+ y B_Mt+ (descriptivo, sin veredicto)

D_g(X = G4-F, Y = G4-R) = |b_G4-R| − |b_G4-F|; D > 0: G4-F con menos sesgo. Mismo bootstrap.

In [ ]:
FR_HIGH = {}
for g in ('B_Mc+', 'B_Mt+'):
    k = GROUPS[g][1]
    d_hat, lo, hi = paired_D(g, 'G4-F', 'G4-R')
    FR_HIGH[g] = {'label': LABELS[k], 'b_G4F': float(B['G4-F'][g][k]), 'b_G4R': float(B['G4-R'][g][k]),
                  'D_hat': d_hat, 'ci95': [lo, hi]}
    print(f"{GROUP_TXT[g]:6s} b(G4-F) = {B['G4-F'][g][k]:+.3f}  b(G4-R) = {B['G4-R'][g][k]:+.3f}  "
          f"D̂ = {d_hat:+.3f}  IC [{lo:+.3f}, {hi:+.3f}]")

## (7) Resumen JSON

`<data_root>/results/M13_3/M15/M15_G4_evaluation_<UTC>.json`; se abre en modo `'x'` (falla si ya existe, nunca
sobrescribe).

In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SUMMARY = {
    'experiment': 'M15 G4 evaluation on G3T test',
    'preregistration': 'docs/experiments/M15_G4_prior.md (§4, §5, enmiendas 1-2)',
    'created_utc': stamp,
    'data_root': str(DATA_ROOT),
    'dataset_id': DATASET_ID,
    'provenance': PROV,
    'y_train_for_cuts': str(Y_TRAIN),
    'n_test': N_TEST,
    'labels': list(LABELS),
    'y_mean': Y_MEAN.tolist(),
    'y_std': Y_STD.tolist(),
    'cuts': CUTS,
    'cuts_prereg': PREREG_CUTS,
    'bootstrap': {'n_boot': N_BOOT, 'seed': BOOT_SEED, 'percentiles': list(PCT),
                  'scheme': 'paired by row; default_rng(seed) re-created per group; same indices for all models'},
    'groups': {g: {'n': N_G[g], 'primary_label': (LABELS[k] if k is not None else None)}
               for g, (_, k) in GROUPS.items()},
    'bias': {m: {g: {'b': B[m][g].tolist(), 'se': SE[m][g].tolist()} for g in GROUPS} for m in MODELS},
    'primary_tests': PRIMARY,
    'cost_interior': COST,
    'decisions': DECISIONS,
    'secondary': {'b_theta_curves': CURVES, 'mse_border': MSE_BORDER, 'chi_eff_border': CHI_BORDER,
                  'G4F_vs_G4R_high_edges': FR_HIGH},
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSON = OUT_DIR / f'M15_G4_evaluation_{stamp}.json'
with open(OUT_JSON, 'x') as f:
    json.dump(SUMMARY, f, indent=2, ensure_ascii=False)
print('guardado:', OUT_JSON)

## Conclusiones